# Generador de código

El requisito: utilizar un modelo de código abierto para generar código C++ de alto rendimiento a partir de código Python

Para replicar esto, deberá configurar un punto final HuggingFace como lo hago en el video. Es fácil de hacer y es bastante satisfactorio ver los resultados.

También es una parte importante de su aprendizaje; este es el primer ejemplo de implementación de un modelo de código abierto detrás de una API. Volveremos a esto en la semana 8, pero esto debería plantar una semilla en su mente sobre lo que implica trasladar modelos de código abierto a producción.

<table style="margin: 0; text-align: left;">
<tr>
<td style="width: 150px; height: 150px; vertical-align: middle;">
<img src="../important.jpg" width="150" height="150" style="display: block;" />
</td>
<td>
<h1 style="color:#900;">Importante: Pausar los puntos finales cuando no estén en uso</h1>
<span style="color:#900;">
Si decide utilizar los puntos finales de HuggingFace para este proyecto, debe detenerlos o pausarlos cuando haya terminado para evitar acumular costos de ejecución innecesarios. Los costos son muy bajos siempre que solo ejecute el punto final cuando lo esté utilizando. Vaya a la interfaz de usuario del punto final de HuggingFace <a href="https://ui.endpoints.huggingface.co/">aquí</a>, abra su punto final y haga clic en Pausar para ponerlo en pausa y no pagar más por él.
Muchas gracias al estudiante John L. por plantear este tema.
<br/><br/>
En la semana 8, usaremos Modal en lugar de puntos finales de HuggingFace; con Modal, solo paga por el tiempo que lo usa y debería obtener créditos gratuitos.
</span>
</td>
</tr>
</table>

In [1]:
# imports

import os
import io
import sys
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic
from IPython.display import Markdown, display, update_display
import gradio as gr
import subprocess

/var/folders/6y/j1c7_h6x7s9504b1v1j36lrm0000gn/T/ipykernel_92199/1757441101.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai


In [2]:
# environment

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')

In [4]:
# initialize

openai = OpenAI()
claude = anthropic.Anthropic()
OPENAI_MODEL = "gpt-5.6-luna"
CLAUDE_MODEL = "claude-opus-5-5"

In [5]:
system_message = "Eres un asistente que reimplementa código Python en C++ de alto rendimiento para una Mac M2. "
system_message += "Responde solo con código C++; usa los comentarios con moderación y no proporciones ninguna explicación más allá de comentarios ocasionales. "
system_message += "La respuesta C++ debe producir una salida idéntica en el menor tiempo posible."

In [6]:
def user_prompt_for(python):
    user_prompt = "Reescribe este código Python en C++ con la implementación más rápida posible que produzca una salida idéntica en el menor tiempo posible."
    user_prompt += "Responde solo con código C++; no expliques tu trabajo más allá de algunos comentarios."
    user_prompt += "Manten la implementación de la generación de números aleatorios idénticos para que los resultados de la coincidencia sean exactos."
    user_prompt += "Responde solo con código C++; no añadas nada más que código; usa los comentarios con moderación y no proporciones ninguna explicación más allá de comentarios ocasionales. "
    user_prompt += "Presta atención a los tipos de números para asegurar que no haya desbordamientos de int (overflow). Recuerda incluir todos los paquetes de C++ necesarios, como iomanip.\n\n"    
    user_prompt += python
    return user_prompt

In [7]:
def messages_for(python):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [8]:
# write to a file called optimized.cpp

def write_output(cpp):
    code = cpp.replace("```cpp","").replace("```","")
    with open("optimized.cpp", "w") as f:
        f.write(code)

In [9]:
def optimize_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        print(fragment, end='', flush=True)
    write_output(reply)

In [10]:
def optimize_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            print(text, end="", flush=True)
    write_output(reply)

In [11]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [12]:
exec(pi)

Result: 3.141592658589
Execution Time: 4.598907 seconds


In [13]:
optimize_gpt(pi)

#include <chrono>
#include <cstdint>
#include <iomanip>
#include <iostream>

static inline double calculate(std::int64_t iterations,
                               std::int64_t param1,
                               std::int64_t param2) {
    double result = 1.0;

    for (std::int64_t i = 1; i <= iterations; ++i) {
        const std::int64_t j1 = i * param1 - param2;
        result -= 1.0 / static_cast<double>(j1);

        const std::int64_t j2 = i * param1 + param2;
        result += 1.0 / static_cast<double>(j2);
    }

    return result;
}

int main() {
    const auto start_time = std::chrono::steady_clock::now();

    const double result = calculate(100'000'000, 4, 1) * 4.0;

    const auto end_time = std::chrono::steady_clock::now();
    const double elapsed =
        std::chrono::duration<double>(end_time - start_time).count();

    std::cout << std::fixed << std::setprecision(12)
              << "Result: " << result << '\n'
              << std::setprecision(6)
              

In [14]:
exec(pi)

Result: 3.141592658589
Execution Time: 4.506643 seconds


In [15]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

Result: 3.141592658589
Execution Time: 0.108040 seconds


In [16]:
optimize_claude(pi)

```cpp
#include <iostream>
#include <iomanip>
#include <chrono>
#include <cstdint>

// No compilar con -ffast-math: el orden de las operaciones en coma flotante debe ser el de Python
static double calculate(int64_t iterations, int64_t param1, int64_t param2) {
    double result = 1.0;
    for (int64_t i = 1; i <= iterations; ++i) {
        int64_t j = i * param1 - param2;
        result -= 1.0 / (double)j;
        j = i * param1 + param2;
        result += 1.0 / (double)j;
    }
    return result;
}

int main() {
    auto start = std::chrono::high_resolution_clock::now();
    double result = calculate(100000000LL, 4, 1) * 4.0;
    auto end = std::chrono::high_resolution_clock::now();
    double elapsed = std::chrono::duration<double>(end - start).count();

    std::cout << std::fixed << std::setprecision(12) << "Result: " << result << "\n";
    std::cout << std::fixed << std::setprecision(6) << "Execution Time: " << elapsed << " seconds\n";
    return 0;
}
```

In [17]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

Result: 3.141592658589
Execution Time: 0.116216 seconds


In [18]:
python_hard = """
def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [19]:
exec(python_hard)

Total Maximum Subarray Sum (20 runs): 10980
Execution Time: 14.969203 seconds


In [20]:
optimize_gpt(python_hard)

#include <chrono>
#include <cstdint>
#include <iomanip>
#include <iostream>

class LCG {
public:
    explicit LCG(std::uint32_t seed) : value(seed) {}

    std::uint32_t next() {
        value = static_cast<std::uint32_t>(
            static_cast<std::uint64_t>(1664525u) * value + 1013904223u
        );
        return value;
    }

private:
    std::uint32_t value;
};

std::int64_t max_subarray_sum(
    int n,
    std::uint32_t seed,
    std::int32_t min_val,
    std::int32_t max_val
) {
    LCG generator(seed);
    const std::uint32_t range =
        static_cast<std::uint32_t>(max_val - min_val + 1);

    std::int64_t current_sum = 0;
    std::int64_t best_sum = 0;

    for (int i = 0; i < n; ++i) {
        const std::int32_t value =
            static_cast<std::int32_t>(generator.next() % range) + min_val;

        if (i == 0) {
            current_sum = value;
            best_sum = value;
        } else {
            current_sum = (current_sum + value > value)
                ? cur

In [21]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

Total Maximum Subarray Sum (20 runs): 10980
Execution Time: 0.000427 seconds


In [22]:
optimize_claude(python_hard)

```cpp
#include <cstdint>
#include <cstdio>
#include <chrono>
#include <vector>
#include <iostream>
#include <iomanip>
#include <algorithm>

static inline uint32_t lcg_next(uint32_t &v) {
    v = 1664525u * v + 1013904223u; // natural wrap mod 2^32
    return v;
}

// Kadane: equivalent to max over all non-empty subarrays (O(n^2) in original)
static int64_t max_subarray_sum(int n, uint32_t seed, int min_val, int max_val) {
    uint32_t v = seed;
    uint32_t range = (uint32_t)(max_val - min_val + 1);
    int64_t best = INT64_MIN;
    int64_t cur = 0;
    bool first = true;
    for (int i = 0; i < n; ++i) {
        int64_t x = (int64_t)(lcg_next(v) % range) + min_val;
        if (first) { cur = x; first = false; }
        else cur = std::max(x, cur + x);
        if (cur > best) best = cur;
    }
    return best;
}

static int64_t total_max_subarray_sum(int n, uint32_t initial_seed, int min_val, int max_val) {
    int64_t total = 0;
    uint32_t g = initial_seed;
    for (int k = 0; k < 

In [23]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

Total Maximum Subarray Sum (20 runs): 10980
Execution Time: 0.000299 seconds


In [24]:
def stream_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace('```cpp\n','').replace('```','')

In [25]:
def stream_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply.replace('```cpp\n','').replace('```','')

In [26]:
def optimize(python, model):
    if model=="GPT":
        result = stream_gpt(python)
    elif model=="Claude":
        result = stream_claude(python)
    else:
        raise ValueError("Modelo Desconocido")
    for stream_so_far in result:
        yield stream_so_far        

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Código en Python:", lines=10, value=python_hard)
        cpp = gr.Textbox(label="C++ code:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude"], label="Selecciona el modelo", value="GPT")
        convert = gr.Button("Convertir código")

    convert.click(optimize, inputs=[python, model], outputs=[cpp])

ui.launch(inbrowser=True)

In [27]:
def execute_python(code):
        try:
            output = io.StringIO()
            sys.stdout = output
            exec(code)
        finally:
            sys.stdout = sys.__stdout__
        return output.getvalue()

In [28]:
def execute_cpp(code):
        write_output(code)
        try:
            compile_cmd = ["clang++", "-Ofast", "-std=c++17", "-march=armv8.5-a", "-mtune=apple-m2", "-mcpu=apple-m2", "-o", "optimized", "optimized.cpp"]
            compile_result = subprocess.run(compile_cmd, check=True, text=True, capture_output=True)
            run_cmd = ["./optimized"]
            run_result = subprocess.run(run_cmd, check=True, text=True, capture_output=True)
            return run_result.stdout
        except subprocess.CalledProcessError as e:
            return f"An error occurred:\n{e.stderr}"

In [29]:
css = """
.python {background-color: #306998;}
.cpp {background-color: #050;}
"""

In [26]:
with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convierte código de Python a C++")
    with gr.Row():
        python = gr.Textbox(label="Código en Python:", value=python_hard, lines=10)
        cpp = gr.Textbox(label="Código en C++:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude"], label="Selecciona el modelo", value="GPT")
    with gr.Row():
        convert = gr.Button("Convertir el código")
    with gr.Row():
        python_run = gr.Button("Ejecutar Python")
        cpp_run = gr.Button("Ejecutar C++")
    with gr.Row():
        python_out = gr.TextArea(label="Resultado en Python:", elem_classes=["python"])
        cpp_out = gr.TextArea(label="Resultado en C++:", elem_classes=["cpp"])

    convert.click(optimize, inputs=[python, model], outputs=[cpp])
    python_run.click(execute_python, inputs=[python], outputs=[python_out])
    cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)

In [30]:
from huggingface_hub import login, InferenceClient
from transformers import AutoTokenizer

In [31]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [32]:
code_qwen = "Qwen/CodeQwen1.5-7B-Chat"
code_gemma = "google/codegemma-7b-it"
CODE_QWEN_URL = "https://mo1u84h30mt5vns6.us-east-1.aws.endpoints.huggingface.cloud"
CODE_GEMMA_URL = "https://c5hggiyqachmgnqg.us-east-1.aws.endpoints.huggingface.cloud"

In [33]:
tokenizer = AutoTokenizer.from_pretrained(code_qwen)
messages = messages_for(pi)
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.46M [00:00<?, ?B/s]

In [34]:
print(text)

<|im_start|>system
Eres un asistente que reimplementa código Python en C++ de alto rendimiento para una Mac M2. Responde solo con código C++; usa los comentarios con moderación y no proporciones ninguna explicación más allá de comentarios ocasionales. La respuesta C++ debe producir una salida idéntica en el menor tiempo posible.<|im_end|>
<|im_start|>user
Reescribe este código Python en C++ con la implementación más rápida posible que produzca una salida idéntica en el menor tiempo posible.Responde solo con código C++; no expliques tu trabajo más allá de algunos comentarios.Manten la implementación de la generación de números aleatorios idénticos para que los resultados de la coincidencia sean exactos.Responde solo con código C++; no añadas nada más que código; usa los comentarios con moderación y no proporciones ninguna explicación más allá de comentarios ocasionales. Presta atención a los tipos de números para asegurar que no haya desbordamientos de int (overflow). Recuerda incluir t

In [35]:
client = InferenceClient(CODE_QWEN_URL, token=hf_token)
stream = client.text_generation(text, stream=True, details=True, max_new_tokens=1000)
for r in stream:
    print(r.token.text, end = "")

HfHubHTTPError: (Request ID: cPkRG0)

403 Forbidden: None.
Cannot access content at: https://mo1u84h30mt5vns6.us-east-1.aws.endpoints.huggingface.cloud.
Make sure your token has the correct permissions.
Forbidden: You don't have the required permissions to complete this action, missing permissions: inference.endpoints.infer.write

In [37]:
def stream_code_qwen(python):
    tokenizer = AutoTokenizer.from_pretrained(code_qwen)
    messages = messages_for(python)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    client = InferenceClient(CODE_QWEN_URL, token=hf_token)
    stream = client.text_generation(text, stream=True, details=True, max_new_tokens=700)
    result = ""
    for r in stream:
        result += r.token.text
        yield result    

In [40]:
def optimize(python, model):
    if model=="GPT":
        result = stream_gpt(python)
    elif model=="Claude":
        result = stream_claude(python)
    elif model=="CodeQwen":
        result = stream_code_qwen(python)
    else:
        raise ValueError("Unknown model")
    for stream_so_far in result:
        yield stream_so_far    

In [27]:
with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convierte código de Python a C++")
    with gr.Row():
        python = gr.Textbox(label="Python code:", value=python_hard, lines=10)
        cpp = gr.Textbox(label="C++ code:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude", "CodeQwen"], label="Selecciona el modelo", value="GPT")
    with gr.Row():
        convert = gr.Button("Convertir el código")
    with gr.Row():
        python_run = gr.Button("Ejecutar Python")
        cpp_run = gr.Button("Ejecutar C++")
    with gr.Row():
        python_out = gr.TextArea(label="Resultado en Python:", elem_classes=["python"])
        cpp_out = gr.TextArea(label="Resultado en C++:", elem_classes=["cpp"])

    convert.click(optimize, inputs=[python, model], outputs=[cpp])
    python_run.click(execute_python, inputs=[python], outputs=[python_out])
    cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)